# Lesson 10: Run an experiment

**You will learn:** how to organize a study as an `Experiment` with conditions, trials and tools, and how to read and replay what it saved.
**Requires:** CPU only, no downloads, under a minute.

A benchmark produces one score. A study often needs more: several conditions (baseline, silenced), repetitions, recordings of internal activity, and a saved record that can be re-analyzed later. That is what `Experiment` is for. Think of it as the lab notebook and the equipment around the subject.

| Term | Meaning |
| --- | --- |
| **Protocol** | The procedure. `SessionProtocol` builds a fresh session for every trial. |
| **Condition** | A setting being compared, such as `baseline` or `silenced`. |
| **Trial** | One independent repetition of a condition. The subject is reset before and after it. |
| **Tool** | Equipment attached to the run: `RecordInputsOutputs`, `RecordActivity`, `Ablate`. |
| **Instrumentation** | The connection that lets tools reach inside the model, such as `TorchInstrumentation`. |

**Session, protocol, benchmark.** A *session* is one exchange of inputs and responses. A *protocol* decides which sessions to run (conditions and trials). A *benchmark* runs sessions and compares the responses with measured data to produce a score.

In [1]:
import warnings
warnings.filterwarnings('ignore')  # keep the lesson output short

In [2]:
import tempfile
import numpy as np
import torch
from brainscore_core import Subject
from brainscore_core.streaming import StreamEvent

torch.manual_seed(0)
net = torch.nn.Sequential(torch.nn.Linear(4, 8), torch.nn.ReLU(), torch.nn.Linear(8, 2)).eval()

class Classifier(Subject):
    identifier = 'toy-classifier'
    in_channels = {'vision'}
    out_channels = {'behavior'}

    def __init__(self, network):
        self.network = network

    def participate(self, session):
        while (event := session.next_input()) is not None:
            with torch.no_grad():
                scores = self.network(torch.as_tensor(event.payload)).numpy()
            session.respond(StreamEvent('behavior', scores, t_ms=event.t_ms))

## The protocol

The session factory receives a `Trial` (with `trial.condition` and `trial.identifier`) and yields a fresh session. Here every trial presents the same input.

In [3]:
from contextlib import contextmanager
from brainscore_core.streaming import BufferedSession
from brainscore.experiments import SessionProtocol

inputs = np.ones(4, dtype=np.float32)  # a 4-number vector stands in for an image

@contextmanager
def make_session(trial):
    yield BufferedSession(inputs=[StreamEvent('vision', inputs.copy(), t_ms=0)],
                          requested_output_channels=['behavior'])

protocol = SessionProtocol(
    identifier='silence-hidden-layer',
    session_factory=make_session,
    conditions=['baseline', 'silenced'],
    trials=[0],
    input_channels=['vision'],
    output_channels=['behavior'],
)

## The experiment

The tools record inputs and outputs, silence layer `'0'` only in the `silenced` condition, and record that layer's activity. Tools are listed in order: recorders first, then interventions, then activity recorded after the intervention.

In [4]:
from brainscore.experiments import Ablate, Experiment, RecordActivity, RecordInputsOutputs, TorchInstrumentation

output_dir = tempfile.mkdtemp() + '/run'
experiment = Experiment(
    subject=Classifier(net),
    protocol=protocol,
    tools=[RecordInputsOutputs(), Ablate(['0'], conditions=['silenced']), RecordActivity(['0'])],
    instrumentation=TorchInstrumentation(net),
    output_dir=output_dir,
)
experiment.validate()   # checks channels and tools without running the model
result = experiment.run()
result.manifest['status']

'complete'

## Read the record

Everything is saved to `output_dir`. `result.record` reads it back without running the model.

In [5]:
for event in result.record.events():
    print(f"{event['condition']:>9}  {event['kind']}")

 baseline  trial_start
 baseline  session_context
 baseline  input
 baseline  activity
 baseline  output
 baseline  trial_end
 silenced  trial_start
 silenced  intervention_start
 silenced  session_context
 silenced  input
 silenced  activity
 silenced  output
 silenced  intervention_end
 silenced  trial_end


In [6]:
for response in result.record.outputs():
    print(response.payload)

[-0.13940221 -0.17760448]
[ 0.06588256 -0.2741119 ]


The two outputs differ because the hidden layer was silenced in the second condition.

## Replay the saved inputs

`replay_sessions` sends the saved inputs to a subject again, condition by condition. `compare_outputs` checks whether the new outputs match the saved ones exactly.

In [7]:
from brainscore.experiments import compare_outputs, replay_sessions

replay = Experiment(
    subject=Classifier(net),
    protocol=replay_sessions(result.directory),
    tools=[RecordInputsOutputs(), Ablate(['0'], conditions=['silenced'])],
    instrumentation=TorchInstrumentation(net),
    output_dir=tempfile.mkdtemp() + '/replay',
).run()
compare_outputs(result.directory, replay.directory)

{'reference_outputs': 2,
 'candidate_outputs': 2,
 'different_indices': [],
 'equal': True,
 'scope': 'exact sequential outputs; not task success or simulator equivalence'}

## Summary

- An `Experiment` = subject + protocol + tools + an output folder.
- **Conditions** are the settings compared; **trials** are independent repetitions with a reset around each.
- Tools such as `Ablate` and `RecordActivity` change or observe the run without changing the subject.
- The saved record can be read (`RunRecord`) and replayed (`replay_sessions`).

You have now seen every core concept. The [experiments](../experiments/) put them to use on scientific questions.